In [1]:
!pip install vllm transformers accelerate



- `nohup` allows you to keep the vLLM server running in the background, even if the terminal is closed.
- `vllm.log` records all server output, and status messages into an external log file for monitoring.


In [2]:
# Start the vLLM server in the background and load the Qwen3-1.7B model
# The server keeps running even if the terminal is closed (nohup + &)

!nohup vllm serve Qwen/Qwen3-1.7B \
  --enable-auto-tool-choice \
  --tool-call-parser hermes \
  --reasoning-parser deepseek_r1 \
  > vllm.log &


nohup: redirecting stderr to stdout


In [6]:
!tail -n 20 vllm.log # to watch last 20 rows og vllm.log, about 2/3 minutes to have the server ready -> Application startup complete


(APIServer pid=1442) INFO 10-29 11:21:31 [launcher.py:42] Route: /v1/responses/{response_id}/cancel, Methods: POST
(APIServer pid=1442) INFO 10-29 11:21:31 [launcher.py:42] Route: /v1/chat/completions, Methods: POST
(APIServer pid=1442) INFO 10-29 11:21:31 [launcher.py:42] Route: /v1/completions, Methods: POST
(APIServer pid=1442) INFO 10-29 11:21:31 [launcher.py:42] Route: /v1/embeddings, Methods: POST
(APIServer pid=1442) INFO 10-29 11:21:31 [launcher.py:42] Route: /pooling, Methods: POST
(APIServer pid=1442) INFO 10-29 11:21:31 [launcher.py:42] Route: /classify, Methods: POST
(APIServer pid=1442) INFO 10-29 11:21:31 [launcher.py:42] Route: /score, Methods: POST
(APIServer pid=1442) INFO 10-29 11:21:31 [launcher.py:42] Route: /v1/score, Methods: POST
(APIServer pid=1442) INFO 10-29 11:21:31 [launcher.py:42] Route: /v1/audio/transcriptions, Methods: POST
(APIServer pid=1442) INFO 10-29 11:21:31 [launcher.py:42] Route: /v1/audio/translations, Methods: POST
(APIServer pid=1442) INFO 10-

In [7]:

from openai import OpenAI

openai_api_key = "EMPTY"
openai_api_base = "http://127.0.0.1:8000/v1"

client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)

model_name = "Qwen/Qwen3-1.7B"

In [18]:
def get_current_temperature(location: str, unit: str = "celsius"):
    """Get current temperature at a location.

    Args:
        location: The location to get the temperature for, in the format "City, State, Country".
        unit: The unit to return the temperature in. Defaults to "celsius". (choices: ["celsius", "fahrenheit"])

    Returns:
        the temperature, the location, and the unit in a dict
    """
    return {
        "temperature": 26.1,
        "location": location,
        "unit": unit,
    }


def get_temperature_date(location: str, date: str, unit: str = "celsius"):
    """Get temperature at a location and date.

    Args:
        location: The location to get the temperature for, in the format "City, State, Country".
        date: The date to get the temperature for, in the format "Year-Month-Day".
        unit: The unit to return the temperature in. Defaults to "celsius". (choices: ["celsius", "fahrenheit"])

    Returns:
        the temperature, the location, the date and the unit in a dict
    """
    return {
        "temperature": 25.9,
        "location": location,
        "date": date,
        "unit": unit,
    }


def get_function_by_name(name):
    if name == "get_current_temperature":
        return get_current_temperature
    if name == "get_temperature_date":
        return get_temperature_date

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_current_temperature",
            "description": "Get current temperature at a location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": 'The location to get the temperature for, in the format "City, State, Country".',
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": 'The unit to return the temperature in. Defaults to "celsius".',
                    },
                },
                "required": ["location"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_temperature_date",
            "description": "Get temperature at a location and date.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": 'The location to get the temperature for, in the format "City, State, Country".',
                    },
                    "date": {
                        "type": "string",
                        "description": 'The date to get the temperature for, in the format "Year-Month-Day".',
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": 'The unit to return the temperature in. Defaults to "celsius".',
                    },
                },
                "required": ["location", "date"],
            },
        },
    },
]
messages = [
    {"role": "user",  "content": "What's the temperature in San Francisco now? How about tomorrow? Current Date: 2024-09-30. Explain your reasoning step by step before giving the final answer "}
]


In [19]:
# send messages and TOOLS to the vLLM server to generate a response

response = client.chat.completions.create(
    model=model_name,
    messages=messages,
    tools=TOOLS,
    temperature=0.7,
    top_p=0.8,
    max_tokens=512,
    extra_body={
        "repetition_penalty": 1.05,
        "chat_template_kwargs": {"enable_thinking": True,
                                 "detailed_thinking": True}
   },
)

In [20]:
import json

messages.append(response.choices[0].message.model_dump())

if tool_calls := messages[-1].get("tool_calls", None):
    for tool_call in tool_calls:
        call_id: str = tool_call["id"]
        if fn_call := tool_call.get("function"):
            fn_name: str = fn_call["name"]
            fn_args: dict = json.loads(fn_call["arguments"])

            fn_res: str = json.dumps(get_function_by_name(fn_name)(**fn_args))

            messages.append({
                "role": "tool",
                "content": fn_res,
                "tool_call_id": call_id,
            })

In [21]:
response = client.chat.completions.create(
    model=model_name,
    messages=messages,
    tools=TOOLS,
    temperature=0.7,
    top_p=0.8,
    max_tokens=512,
    extra_body={
        "repetition_penalty": 1.05,
        "chat_template_kwargs": {"enable_thinking": True,
                                 "detailed_thinking": True}
   },
)

messages.append(response.choices[0].message.model_dump())

In [22]:
print(response.choices[0].message.content)



The current temperature in San Francisco is **26.1°C**. Tomorrow (October 1, 2024) it will be **25.9°C**. 

### Reasoning:
1. **Current Temperature**: Used `get_current_temperature` with the location "San Francisco, CA, US" to fetch the immediate temperature.
2. **Tomorrow's Temperature**: Used `get_temperature_date` with the date "2024-10-01" (next day after 2024-09-30) to retrieve the temperature for the following day. Both results are in Celsius.


In [23]:
response.choices[0].message.model_dump()

{'content': '\n\nThe current temperature in San Francisco is **26.1°C**. Tomorrow (October 1, 2024) it will be **25.9°C**. \n\n### Reasoning:\n1. **Current Temperature**: Used `get_current_temperature` with the location "San Francisco, CA, US" to fetch the immediate temperature.\n2. **Tomorrow\'s Temperature**: Used `get_temperature_date` with the date "2024-10-01" (next day after 2024-09-30) to retrieve the temperature for the following day. Both results are in Celsius.',
 'refusal': None,
 'role': 'assistant',
 'annotations': None,
 'audio': None,
 'function_call': None,
 'tool_calls': [],
 'reasoning_content': '\nOkay, let me try to figure out how to answer the user\'s question. They asked for the current temperature in San Francisco and the temperature tomorrow, with the current date being September 30, 2024.\n\nFirst, I need to check the available functions. There\'s get_current_temperature which takes a location and unit, and get_temperature_date which requires location, date, an